In [8]:
import sys
import os
import mysql.connector
import pandas as pd
sys.path.append(os.path.abspath('..'))
from config import get_db_config
import warnings
warnings.filterwarnings('ignore')
import json
import pickle
import numpy as np
import datetime
from IPython.display import display, HTML

In [9]:
# Connect ke database config
config = get_db_config()
# Ambil host dari salah satu config (misal db_old)
print(f'Database config loaded: {config["db_old"]["host"]}')

# Connect ke DB Lama
db_old = mysql.connector.connect(**config['db_old'])
cursor_old = db_old.cursor(dictionary=True)
print(f'Connected to old database: {config["db_old"]["database"]}')

# Connect ke DB Baru
db_new = mysql.connector.connect(**config['db_new'])
cursor_new = db_new.cursor(dictionary=True)
print(f'Connected to new database: {config["db_new"]["database"]}')


Database config loaded: localhost
Connected to old database: dataleap_v5_example
Connected to new database: dataleap_v5_migration


In [10]:
print("================================================================================")
print(" 🚀 SETUP INSERT HANDLER GLOBAL - FASE 2 (SALING SILANG & AUTO-SKIP) 🚀 ")
print("================================================================================")

# ================================================================================
# TAHAP 1: MEMUAT SE LURUH BERKAS PICKLE DAN MERGE JADI SATU PINTU
# ================================================================================
all_fase_2_data = {}

# 1. Load File Cimut
try:
    with open('fase_2_cimut.pkl', 'rb') as f:
        all_fase_2_data.update(pickle.load(f))
    print("✓ Berhasil memuat data hasil konversi Cimut.")
except Exception as e:
    print(f"⚠️ Gagal memuat file pkl Cimut: {e}")

# 2. Load File Afrida
try:
    with open('fase_2_afrida.pkl', 'rb') as f:
        all_fase_2_data.update(pickle.load(f))
    print("✓ Berhasil memuat data hasil konversi Afrida.")
except Exception as e:
    print(f"⚠️ Gagal memuat file pkl Afrida: {e}")

# 3. Load File Hanif (Jika ada file terpisah, opsional)
try:
    if os.path.exists('fase_2_hanif.pkl'):
        with open('fase_2_hanif.pkl', 'rb') as f:
            all_fase_2_data.update(pickle.load(f))
        print("✓ Berhasil memuat data hasil konversi Hanif.")
except:
    pass

 🚀 SETUP INSERT HANDLER GLOBAL - FASE 2 (SALING SILANG & AUTO-SKIP) 🚀 
✓ Berhasil memuat data hasil konversi Cimut.
✓ Berhasil memuat data hasil konversi Afrida.


In [11]:
# ================================================================================
# TAHAP 2: ATUR URUTAN STRATEGIS SALING SILANG (KARYA CIMUT & AFRIDA)
# ================================================================================
# Susun urutannya di sini secara mutlak, bebas saling silang antar tim!
tables_to_insert_ordered = [
    # --- Blok Awal: Data Induk Fondasi (Milik Cimut) ---
    'periode','parameter_nilai', 'karyawan', 'bidang_kategori', 'keluarga_karyawan', 'bidang_link'   
]

In [12]:
import pandas as pd
import datetime
import numpy as np
import mysql.connector

# ================================================================================
# TAHAP 3: FUNGSI UTAMA INSERT DENGAN RINGKASAN DI ATAS & DIAGNOSTIK ERROR DI BAWAH
# ================================================================================
def insert_data_with_preview_and_skip_v2(db_connection, cursor, tables_data, ordered_list):
    results = {}
    
    print("="*80)
    print("🎬 MEMULAI EKSEKUSI PENYUNTIKAN DATA KE MYSQL BARU (SISTEM RINGKASAN ATAS)")
    print("="*80)
    
    # ----------------------------------------------------------------------------
    # SUB-LANGKAH A: PROSES INSERT KE MYSQL DI BELAKANG LAYAR
    # ----------------------------------------------------------------------------
    for table_name in ordered_list:
        if table_name not in tables_data:
            results[table_name] = {
                'status': 'not_found', 
                'rows': 0, 
                'msg': f'⚠️  {table_name}: Tidak ditemukan di file pkl'
            }
            continue
            
        df_target = tables_data[table_name]
        
        if df_target is None or df_target.empty:
            results[table_name] = {
                'status': 'empty', 
                'rows': 0, 
                'msg': f'ℹ️  {table_name}: DataFrame kosong (0 baris)'
            }
            continue
            
        try:
            # Bersihkan kolom kosong murni agar tidak merusak placeholder query
            df_to_push = df_target.dropna(axis=1, how='all')
            
            columns_str = ', '.join([f'`{col}`' for col in df_to_push.columns])
            placeholders_str = ', '.join(['%s'] * len(df_to_push.columns))
            
            # Gunakan INSERT IGNORE untuk auto-skip duplikat primary key
            insert_query = f"INSERT IGNORE INTO `{table_name}` ({columns_str}) VALUES ({placeholders_str})"
            
            # Konversi DataFrame ke Native List Python (Hancurkan tipe data NumPy)
            raw_numpy_list = df_to_push.to_numpy().tolist()
            clean_data_tuples = [
                tuple(None if pd.isna(x) or str(x).strip() in ["NaT", "NaN"] else x for x in row) 
                for row in raw_numpy_list
            ]
            
            # Eksekusi massal
            cursor.executemany(insert_query, clean_data_tuples)
            db_connection.commit()
            
            total_rows = len(clean_data_tuples)
            results[table_name] = {
                'status': 'success', 
                'rows': total_rows, 
                'msg': f'✓ {table_name}: Sukses diproses! Sebanyak {total_rows} baris sukses dimasukkan / di-skip aman.'
            }
            
        except Exception as e:
            db_connection.rollback()
            results[table_name] = {
                'status': 'failed', 
                'rows': 0, 
                'msg': f'✗ {table_name}: Gagal total saat insert - Alasan: {e}'
            }

    # ----------------------------------------------------------------------------
    # 📊 CETAK PAPAN RINGKASAN DI PALING ATAS (SUMMARY BOARD)
    # ----------------------------------------------------------------------------
    print("\n================================================================================")
    print(" 📊 PAPAN RINGKASAN STATUS MIGRATION DATA (SUMMARY BOARD) 📊")
    print("================================================================================")
    
    # 1. Cetak yang sukses dulu biar rapi
    print("🟢 TABEL YANG SUKSES MASUK:")
    success_exist = False
    for table_name in ordered_list:
        if table_name in results and results[table_name]['status'] == 'success':
            print(f"  {results[table_name]['msg']}")
            success_exist = True
    if not success_exist: print("  (Tidak ada tabel yang sukses)")

    print("\n🔴 TABEL YANG BERMASALAH / GAGAL MASUK (MOHON DIANALISA):")
    failed_exist = False
    for table_name in ordered_list:
        if table_name in results and results[table_name]['status'] in ['failed', 'not_found', 'empty']:
            print(f"  {results[table_name]['msg']}")
            failed_exist = True
    if not failed_exist: print("  🎉 LUAR BIASA! Semua tabel bersih tidak ada yang gagal.")
            
    print("================================================================================\n")


    # ----------------------------------------------------------------------------
    # 📸 CETAK PREVIEW HISTORI & DIAGNOSTIK ERROR DI BAGIAN BAWAH
    # ----------------------------------------------------------------------------
    print("="*80)
    print("📸 MEMULAI LOG VISUALISASI PREVIEW & DIAGNOSTIK TABEL")
    print("="*80)
    
    for table_name in ordered_list:
        if table_name in results:
            # Taktik A: Jika Sukses, tampilkan preview standard 5 baris teratas
            if results[table_name]['status'] == 'success':
                print(f"\n📂 [🟢 PREVIEW TABEL SUKSES: {table_name.upper()}]")
                print("-" * 50)
                display(tables_data[table_name]) 
                print("-" * 80)
                
            # Taktik B: Jika GAGAL, tembak dan kuliti struktur datanya secara transparan!
            elif results[table_name]['status'] == 'failed':
                print(f"\n🚨 [🔴 DIAGNOSTIK TABEL ERROR: {table_name.upper()}] 🚨")
                print(f"Alasan MySQL Menolak: {results[table_name]['msg']}")
                print("-" * 50)
                print("Berikut 5 baris sampel data yang gagal dikirim, cek tipe datanya (apakah ada .0 atau string aneh):")
                display(tables_data[table_name])
                print(f"\nTipe data kolom internal DataFrame untuk tabel '{table_name}':")
                # Menampilkan tipe data internal pandas agar ketahuan mana float ghaib / objek aneh
                print(tables_data[table_name].dtypes)
                print("-" * 80)
            
    print("\n" + "="*80)
    print("🏁 PROSES INSPEKSI SELESAI. SILAKAN CEK HASIL DIAGNOSTIK DI ATAS 🏁")
    print("="*80)
    return results

In [13]:
# ================================================================================
# TAHAP 4: MENJALANKAN EKSEKUSI DATA REAL (MENGGUNAKAN VERSI RINGKASAN ATAS)
# ================================================================================
results_fase_2 = insert_data_with_preview_and_skip_v2(
    db_connection=db_new, 
    cursor=cursor_new, 
    tables_data=all_fase_2_data, 
    ordered_list=tables_to_insert_ordered
)

🎬 MEMULAI EKSEKUSI PENYUNTIKAN DATA KE MYSQL BARU (SISTEM RINGKASAN ATAS)

 📊 PAPAN RINGKASAN STATUS MIGRATION DATA (SUMMARY BOARD) 📊
🟢 TABEL YANG SUKSES MASUK:
  ✓ periode: Sukses diproses! Sebanyak 91 baris sukses dimasukkan / di-skip aman.
  ✓ parameter_nilai: Sukses diproses! Sebanyak 1187 baris sukses dimasukkan / di-skip aman.
  ✓ karyawan: Sukses diproses! Sebanyak 51 baris sukses dimasukkan / di-skip aman.
  ✓ bidang_kategori: Sukses diproses! Sebanyak 12 baris sukses dimasukkan / di-skip aman.
  ✓ keluarga_karyawan: Sukses diproses! Sebanyak 64 baris sukses dimasukkan / di-skip aman.
  ✓ bidang_link: Sukses diproses! Sebanyak 7 baris sukses dimasukkan / di-skip aman.

🔴 TABEL YANG BERMASALAH / GAGAL MASUK (MOHON DIANALISA):
  🎉 LUAR BIASA! Semua tabel bersih tidak ada yang gagal.

📸 MEMULAI LOG VISUALISASI PREVIEW & DIAGNOSTIK TABEL

📂 [🟢 PREVIEW TABEL SUKSES: PERIODE]
--------------------------------------------------


,id_periode,nama_periode,id_kursus,jumlah_sesi,tahun_ajar,tanggal_mulai,status,is_active
0,P00006,General English Term I July-October 2023,K00001,30,2023/2024,2023-07-04,1,1
1,P00008,General English Term II Oct '23 - Feb '24,K00001,30,2023/2024,2023-10-25,1,1
2,P00009,General English Term III Feb-Jun 2024,K00001,30,2023/2024,2024-02-21,1,1
3,P00010,Coding Semester I-2023/2024,K00002,18,2023/2024,2023-08-01,1,1
4,P00011,Coding Semester II-2023/2024,K00002,18,2023/2024,2024-01-23,1,1
...,...,...,...,...,...,...,...,...
87,P00102,RUPIN BSI Program Komputer JUL-DES 2026,K00020,43,2025/2026,2026-07-02,1,1
88,P00103,Kemitraan - TK MITRA MJ JAN-MEI 2026,K00021,34,2025/2026,2026-01-05,1,1
90,P00105,Kemitraan - CC MITRA MJ JAN-MEI 2026,K00022,34,2025/2026,2026-01-05,1,1
91,P00106,Kemitraan - Ekskul Coding (Al Muslim),K00012,30,2026,2026-08-01,1,1


--------------------------------------------------------------------------------

📂 [🟢 PREVIEW TABEL SUKSES: PARAMETER_NILAI]
--------------------------------------------------


,id_level,nama_parameter,status_parameter
0,L00022,Class participation,0
1,L00022,Oral,0
2,L00022,Listening,0
3,L00022,Writing,0
4,L00022,Writing-1,1
...,...,...,...
1182,L00184,Grade,0
1183,L00184,Comments,0
1184,L00185,listening,1
1185,L00185,speaking,1


--------------------------------------------------------------------------------

📂 [🟢 PREVIEW TABEL SUKSES: KARYAWAN]
--------------------------------------------------


...

--------------------------------------------------------------------------------

📂 [🟢 PREVIEW TABEL SUKSES: KELUARGA_KARYAWAN]
--------------------------------------------------


,id_keluarga,id_karyawan,hubungan_keluarga,nama_lengkap,pekerjaan,nomor_hp
0,3,LEAP00313III23,Ibu,Nur Fatmawati,Wiraswasta,081234477137
1,4,LEAP01901VI19,Suami,Fathul Kirom,Suami,0852-3101-7799
2,5,LEAP02602VIII21,Ayah,"Suwandi, S.Pd.",Guru Matematika SMAN 17 Surabaya,087853591616
3,6,LEAP02602VIII21,Ibu,"Ir. Hj. Erhasyati Islamiyah, M.M.",(Pensiun) Guru Biologi SMA Muhammadiyah 2 Sura...,08179365966
4,7,LEAP01101XII20,Suami,JONATHAN O'DRISCOLL,TIDAK BEKERJA,089696320278
...,...,...,...,...,...,...
59,65,LEAP05431VII24,Ibu,Sylvia Widyastuti,Pekerja swasta,082142995636
60,66,LEAP06826I26,Ibu,Siti Rokhani,Kerja di toko kelontong,085775734255
61,67,LEAP06915IV26,Ibu,Ratri Widorini,School Accounting,+62 822-6433-4296
62,68,LEAP06915IV26,Ayah,Agung Yuniarti Akhirin,Accounting,+62 813-3022-2722


--------------------------------------------------------------------------------

📂 [🟢 PREVIEW TABEL SUKSES: BIDANG_LINK]
--------------------------------------------------


,id_bidang_link,nama_form,link_drive,id_bidang_kategori,status_share
0,16,Daftar Training,https://drive.google.com/drive/folders/1BCPhp6...,13,0
1,17,Referensi,https://drive.google.com/drive/folders/1Lo6hzu...,14,0
2,18,Dokumentasi,https://drive.google.com/drive/folders/14ieegP...,24,0
3,19,List Training,https://drive.google.com/drive/folders/1yivPYk...,16,0
4,20,Referensi,https://drive.google.com/drive/folders/1ah2G3X...,19,0
5,21,Logo Leap,https://drive.google.com/drive/folders/1S1Og9c...,7,0
6,22,Referensi,https://drive.google.com/drive/folders/1JuWEPT...,20,0


--------------------------------------------------------------------------------

🏁 PROSES INSPEKSI SELESAI. SILAKAN CEK HASIL DIAGNOSTIK DI ATAS 🏁


In [14]:
# print("================================================================================")
# print(" 🧹 MEMULAI PROSES TRUNCATE DATA GLOBAL - FASE 2 (SISTEM RINGKASAN ATAS) 🧹 ")
# print("================================================================================")

# def truncate_tables_with_summary(db_connection, cursor, ordered_list):
#     truncate_results = {}
    
#     try:
#         # 🔥 SAKTI 1: Matikan benteng Foreign Key checks agar MySQL tidak memblokir penghapusan
#         cursor.execute("SET FOREIGN_KEY_CHECKS=0")
#         db_connection.commit()
#         print("🔓 Sensor Foreign Key Checks berhasil DIMATIKAN sementara.\n")
#     except Exception as e:
#         print(f"✗ Gagal mematikan Foreign Key Checks: {e}")
#         return
        
#     # ----------------------------------------------------------------------------
#     # SUB-LANGKAH A: PROSES EKSEKUSI TRUNCATE DI BELAKANG LAYAR
#     # ----------------------------------------------------------------------------
#     for table_name in ordered_list:
#         try:
#             truncate_query = f"TRUNCATE TABLE `{table_name}`"
#             cursor.execute(truncate_query)
#             db_connection.commit()
            
#             truncate_results[table_name] = {
#                 'status': 'success',
#                 'msg': f"✓ {table_name}: Sukses dibersihkan total! Seluruh baris data amblas."
#             }
#         except Exception as e:
#             db_connection.rollback()
#             truncate_results[table_name] = {
#                 'status': 'failed',
#                 'msg': f"✗ {table_name}: Gagal dikosongkan! Alasan: {e}"
#             }

#     try:
#         # 🔥 SAKTI 2: Wajib nyalakan kembali benteng Foreign Key checks setelah selesai
#         cursor.execute("SET FOREIGN_KEY_CHECKS=1")
#         db_connection.commit()
#         print("🔒 Sensor Foreign Key Checks berhasil DIHIDUPKAN kembali dengan aman.")
#     except Exception as e:
#         print(f"⚠️ Peringatan: Gagal menghidupkan kembali Foreign Key Checks: {e}")

#     # ----------------------------------------------------------------------------
#     # 🔥 CETAK PAPAN RINGKASAN TRUNCATE DI PALING ATAS (ANTI-SCROLL BOARD)
#     # ----------------------------------------------------------------------------
#     print("\n================================================================================")
#     print(" 📊 PAPAN RINGKASAN STATUS TRUNCATE DATABASE (CLEANUP SUMMARY BOARD) 📊")
#     print("================================================================================")
#     for table_name in ordered_list:
#         if table_name in truncate_results:
#             print(truncate_results[table_name]['msg'])
#         else:
#             print(f"⚠️  {table_name}: Lewat dari antrean pembersihan.")
#     print("================================================================================")
    
#     return truncate_results

# # === JALANKAN EKSEKUSI PEMBERSIHAN MENGGUNAKAN DAFTAR TABEL SALING SILANGMU ===
# results_truncate_fase_2 = truncate_tables_with_summary(
#     db_connection=db_new, 
#     cursor=cursor_new, 
#     ordered_list=tables_to_insert_ordered  # Otomatis memakai list urutan saling silang yang kita buat tadi
# )